In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    initcap,
    current_timestamp,
    when
)


BRONZE_SCHEMA = "workspace.bronze"
SILVER_SCHEMA = "workspace.silver"

TABLES = [
    "artistas",
    "albuns",
    "musicas",
    "usuarios",
    "reproducoes"
]


In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}
""")

results_silver = []

In [0]:
artistas = (
    spark.table(f"{BRONZE_SCHEMA}.artistas")
        .select(
            col("id").cast("string").alias("id"),
            trim(initcap(col("nome"))).alias("nome"),
            trim(initcap(col("pais"))).alias("pais"),
            upper(trim(col("genero"))).alias("genero"),
            col("created_at").alias("criado_em")
        )
        .dropDuplicates(["id"])
        .filter("id IS NOT NULL")
        .filter("nome IS NOT NULL")
        .withColumn("_silver_processed_at", current_timestamp())
)

(
    artistas.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{SILVER_SCHEMA}.artistas")
)

results_silver.append({
    "table": "artistas",
    "rows": artistas.count()
})


In [0]:
albuns = (
    spark.table(f"{BRONZE_SCHEMA}.albuns")
        .select(
            col("id").cast("string").alias("id"),
            col("artista_id").cast("string").alias("artista_id"),
            trim(col("titulo")).alias("titulo"),
            col("ano_lancamento").cast("int").alias("ano_lancamento"),
            col("total_faixas").cast("int").alias("total_faixas"),
            col("created_at").alias("criado_em")
        )
        .dropDuplicates(["id"])
        .filter("id IS NOT NULL")
        .filter("artista_id IS NOT NULL")
        .filter("titulo IS NOT NULL")
        .filter("ano_lancamento > 1900")
        .filter("total_faixas > 0")
        .withColumn("_silver_processed_at", current_timestamp())
)

(
    albuns.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{SILVER_SCHEMA}.albuns")
)

results_silver.append({
    "table": "albuns",
    "rows": albuns.count()
})

In [0]:
musicas = (
    spark.table(f"{BRONZE_SCHEMA}.musicas")
        .select(
            col("id").cast("string").alias("id"),
            col("album_id").cast("string").alias("album_id"),
            trim(col("titulo")).alias("titulo"),
            col("duracao_segundos").cast("int").alias("duracao_segundos"),
            col("numero_faixa").cast("int").alias("numero_faixa"),
            col("created_at").alias("criado_em")
        )
        .dropDuplicates(["id"])
        .filter("id IS NOT NULL")
        .filter("album_id IS NOT NULL")
        .filter("titulo IS NOT NULL")
        .filter("duracao_segundos > 0")
        .filter("numero_faixa > 0")
        .withColumn("_silver_processed_at", current_timestamp())
)

(
    musicas.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{SILVER_SCHEMA}.musicas")
)

results_silver.append({
    "table": "musicas",
    "rows": musicas.count()
})

In [0]:
# USUARIOS
usuarios = (
    spark.table(f"{BRONZE_SCHEMA}.usuarios")
        .select(
            col("id").cast("string").alias("id"),
            trim(initcap(col("nome"))).alias("nome"),
            trim(lower(col("email"))).alias("email"),
            trim(lower(col("plano"))).alias("plano"),
            col("created_at").alias("criado_em")
        )
        .dropDuplicates(["id"])
        .dropDuplicates(["email"])
        .filter("id IS NOT NULL")
        .filter("nome IS NOT NULL")
        .filter("email IS NOT NULL")
        .withColumn("_silver_processed_at", current_timestamp())
)

(
    usuarios.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{SILVER_SCHEMA}.usuarios")
)

results_silver.append({
    "table": "usuarios",
    "rows": usuarios.count()
})

In [0]:
reproducoes = (
    spark.table(f"{BRONZE_SCHEMA}.reproducoes")
        .select(
            col("id").cast("string").alias("id"),
            col("usuario_id").cast("string").alias("usuario_id"),
            col("musica_id").cast("string").alias("musica_id"),
            col("reproduzida_em"),
            col("duracao_ouvida_segundos")
                .cast("int")
                .alias("duracao_ouvida_segundos"),
            col("created_at").alias("criado_em")
        )
        .dropDuplicates(["id"])
        .filter("id IS NOT NULL")
        .filter("usuario_id IS NOT NULL")
        .filter("musica_id IS NOT NULL")
        .filter("duracao_ouvida_segundos >= 0")
        .withColumn(
            "reproducao_completa",
            when(
                col("duracao_ouvida_segundos") >= 180,
                True
            ).otherwise(False)
        )
        .withColumn("_silver_processed_at", current_timestamp())
)

(
    reproducoes.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{SILVER_SCHEMA}.reproducoes")
)

results_silver.append({
    "table": "reproducoes",
    "rows": reproducoes.count()
})

In [0]:
print("Camada Silver criada com sucesso.")

In [0]:
for item in results_silver:
    print(f"{item['table']}: {item['rows']} registros")